<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Generate_Better_Synthetic_Dialogues_with_a_User_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

More details in this article: [Generate Better Synthetic Datasets with a "User" LLM](https://kaitchup.substack.com/p/generate-better-synthetic-datasets)


This notebook implements the setup recommended by the UserLM paper (arXiv:2510.06552), so the *user* model behaves like **user**:
- User turns are **conditioned on a high-level intent** and the dialogue state.
- Utterances are **short** (3–25 words) and may **spread information across turns**.
- The user can **end** the conversation using `<|endconversation|>`.

**Models**
- **User (simulator):** `microsoft/UserLM-8b` (CPU by default)
- **Assistant:** `Qwen/Qwen3-4B-Instruct-2507` (GPU via vLLM)

> 💡 Why this layout? Loading *two* engines on a single Colab GPU often OOMs. We keep the **assistant on GPU** and run the **user on CPU** with Transformers.

## Installation

Installs vLLM and Transformers

In [ ]:
!pip install --upgrade vllm transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 160.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 140.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 155.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.


Assistant on **GPU via vLLM**; User on **CPU** (Transformers). In theory, this is possible to serve both with vLLM using a single GPU but it couldn't run with Colab.

Ideally, the two models should be served from two difference devices.

Also note that the conversation will be cut-off once the max_model_len is reached, as in the logs below this cell.

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc, re, traceback
import torch

ASSISTANT_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
USER_MODEL_ID      = "microsoft/UserLM-8b"

USER_EOT = "<|eot_id|>"
USER_ENDCONV = "<|endconversation|>"

MAX_TURNS = 20
MAX_NEW_ASST = 500
MAX_NEW_USER = 500

tok_user = AutoTokenizer.from_pretrained(USER_MODEL_ID, trust_remote_code=True)
tok_asst = AutoTokenizer.from_pretrained(ASSISTANT_MODEL_ID, trust_remote_code=True)

# Load Assistant model on GPU using vLLM
assistant_llm = LLM(
    model=ASSISTANT_MODEL_ID,
    gpu_memory_utilization=0.4,
    max_model_len=2000,
)

# Load User model on CPU using Transformers
user_model = AutoModelForCausalLM.from_pretrained(USER_MODEL_ID, trust_remote_code=True)
user_model.to("cpu")


def serialize(tok, messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# Sampling
asst_sampling = SamplingParams(
    temperature=0.7, top_p=0.9, max_tokens=MAX_NEW_ASST, stop=[USER_EOT]
)
user_sampling = SamplingParams(
    temperature=1.0, top_p=0.8, max_tokens=MAX_NEW_USER, stop=[USER_EOT, USER_ENDCONV]
)

# Conversation states
USER_TASK_INTENT = "You are a user chatting with an assistant language model to get information about strategies for losing weight and the impact of certain drugs or medications on weight gain."
user_msgs = [{"role": "system", "content": USER_TASK_INTENT}]
asst_msgs = [{"role": "system", "content": "You are a helpful, concise assistant."}]
next_role = "user"

def clean_user_text(t):
    t = t.strip()
    t = re.sub(r"^(user|assistant)\s*:\s*", "", t, flags=re.I)
    return t.strip()

print("\n================ CONVERSATION ================\n")

for _ in range(MAX_TURNS):
    if next_role == "user":
        # Use the loaded user model
        prompt = serialize(tok_user, user_msgs)
        inputs = tok_user.apply_chat_template(user_msgs, return_tensors="pt").to("cpu") # Move inputs to CPU
        outputs = user_model.generate(
            input_ids=inputs,
            do_sample=True,
            top_p=user_sampling.top_p,
            temperature=user_sampling.temperature,
            max_new_tokens=user_sampling.max_tokens,
            eos_token_id=tok_user.encode(USER_EOT, add_special_tokens=False)[0],
            pad_token_id=tok_user.eos_token_id,
            min_new_tokens=15, # Very important, otherwise UserLLM will generate nothing very often
            bad_words_ids=[[token_id] for token_id in tok_user.encode(USER_ENDCONV, add_special_tokens=False)]
        )

        out = tok_user.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True).strip()




        out = clean_user_text(out)
        print(f"[User]: {out}\n")
        user_msgs.append({"role": "assistant", "content": out})
        asst_msgs.append({"role": "user", "content": out})
        next_role = "assistant"
    else:
        # Use the loaded assistant model
        prompt = serialize(tok_asst, asst_msgs)
        out = assistant_llm.generate([prompt], asst_sampling)[0].outputs[0].text.strip()
        print(f"[Assistant]: {out}\n")
        asst_msgs.append({"role": "assistant", "content": out})
        user_msgs.append({"role": "user", "content": out})
        next_role = "user"

print("==================== END ====================\n")

INFO 10-21 08:31:57 [__init__.py:216] Automatically detected platform cuda.
INFO 10-21 08:32:06 [utils.py:233] non-default args: {'max_model_len': 2000, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}
INFO 10-21 08:32:07 [model.py:547] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 10-21 08:32:07 [model.py:1510] Using max model len 2000
INFO 10-21 08:32:10 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 10-21 08:32:10 [__init__.py:3036] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 10-21 08:32:55 [llm.py:306] Supported_tasks: ['generate']


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


...

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[Assistant]: Thank you for sharing the article. Let's address the **claim** in the article — *"Cannabis may help reduce weight"* — with **scientific accuracy, clarity, and responsibility**.

---

### ❌ Does Cannabis Actually Help with Weight Loss?

**Short answer: No — not reliably or safely. The evidence is weak, misleading, and context-dependent.**

---

### 🔍 What the Article Says (and Why It’s Misleading)

The article cites a **study from the University of Miami** (published in *Medical Cannabis and Cannabinoids*) showing that patients with chronic pain who used cannabis-based medications **lost an average of 4.5 kg** over time — compared to 1 kg in the placebo group.

But here's the **critical context** that the article omits or underplays:

#### 1. **The study does not prove cannabis causes weight loss**
- The patients were already **on medical cannabis for chronic pain**, not for weight loss.
- They may have lost weight due to **reduced pain**, which led to:
  - Improved mobilit

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: The decoder prompt (length 2522) is longer than the maximum model length of 2000. Make sure that `max_model_len` is no smaller than the number of text tokens.